# Ray Serve: Scalable ML Model Serving

## What Is Ray Serve?

Imagine you open a popular coffee shop (ML model serving). One barista (server) can't handle 10,000 customers.  
You hire more baristas and have them work in parallel.  
**Ray Serve** is the system that manages all these baristas — adds more when busy, removes when quiet, and routes customers efficiently.

**Ray Serve** is a scalable model serving library built on **Ray** (a distributed computing framework).  
Key capabilities:
- Scale from 1 to 1000 replicas with one config change
- GPU-aware: request GPU allocation per replica
- Compose models: chain multiple models in a pipeline
- Autoscaling: automatically adjust replicas based on traffic
- Works with FastAPI for HTTP handling

## Resources

- **Docs**: [https://docs.ray.io/en/latest/serve/](https://docs.ray.io/en/latest/serve/)
- **GitHub**: [https://github.com/ray-project/ray](https://github.com/ray-project/ray)
- **YouTube — Ray Serve**: [https://www.youtube.com/watch?v=mM4hJLelzSw](https://www.youtube.com/watch?v=mM4hJLelzSw)

## Installation

```bash
pip install 'ray[serve]'
# Includes Ray core + Serve + FastAPI integration
```

In [ ]:
import numpy as np
import json, time
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

try:
    import ray
    from ray import serve
    RAY_AVAILABLE = True
    print(f"Ray version: {ray.__version__}")
except ImportError:
    RAY_AVAILABLE = False
    print("Ray not installed — simulated output shown. Install: pip install 'ray[serve]'")

# Train two models for composition demo
np.random.seed(42)
X, y = make_classification(n_samples=2000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train)
X_te_s  = scaler.transform(X_test)

fast_model = RandomForestClassifier(n_estimators=10, random_state=42)
fast_model.fit(X_tr_s, y_train)

accurate_model = GradientBoostingClassifier(n_estimators=200, random_state=42)
accurate_model.fit(X_tr_s, y_train)

print(f"Fast model accuracy:     {fast_model.score(X_te_s, y_test):.3f}")
print(f"Accurate model accuracy: {accurate_model.score(X_te_s, y_test):.3f}")

## Core Concept 1: Deployments

A **Deployment** is a class decorated with `@serve.deployment`.  
Each deployment has:
- `num_replicas`: how many copies to run in parallel
- `ray_actor_options`: CPU/GPU allocation per replica
- `autoscaling_config`: min/max replicas, scale-up/down triggers

In [ ]:
if RAY_AVAILABLE:
    # Initialize Ray (local cluster)
    if not ray.is_initialized():
        ray.init(ignore_reinit_error=True, num_cpus=2)
    serve.start(detached=False)

    @serve.deployment(
        num_replicas=2,                      # run 2 copies
        ray_actor_options={"num_cpus": 0.5}, # 0.5 CPU per replica
    )
    class FastModelDeployment:
        def __init__(self):
            """Called once per replica at startup. Load model here."""
            self.model = fast_model
            self.scaler = scaler
            print("FastModel replica initialized")

        async def __call__(self, request):
            """Called for each HTTP request."""
            data = await request.json()
            features = np.array(data['features']).reshape(1, -1)
            features_s = self.scaler.transform(features)
            prediction = int(self.model.predict(features_s)[0])
            proba = self.model.predict_proba(features_s)[0].tolist()
            return {"prediction": prediction, "confidence": max(proba), "model": "fast"}

    # Deploy
    fast_deployment = FastModelDeployment.bind()
    handle = serve.run(fast_deployment, name="fast-model", route_prefix="/fast")

    # Test with HTTP request
    import requests
    try:
        resp = requests.post(
            "http://localhost:8000/fast",
            json={"features": X_test[0].tolist()},
            timeout=5
        )
        print(f"Response: {resp.json()}")
    except Exception as e:
        # Also works via Ray handle
        result = ray.get(handle.remote({"features": X_test[0].tolist()}))
        print(f"Handle result: {result}")

    serve.shutdown()
    ray.shutdown()

else:
    print("Ray Serve deployment (simulated):")
    print()
    print("  @serve.deployment(")
    print("      num_replicas=2,")
    print("      ray_actor_options={'num_cpus': 0.5, 'num_gpus': 0},")
    print("  )")
    print("  class FastModelDeployment:")
    print("      def __init__(self):")
    print("          self.model = load_model()  # runs once per replica")
    print()  
    print("      async def __call__(self, request):")
    print("          data = await request.json()")
    print("          return model.predict(data['features'])")
    print()
    print("  # Deploy and get a handle")
    print("  handle = serve.run(FastModelDeployment.bind(), name='fast-model')")
    print()
    print("  Simulated response: {prediction: 1, confidence: 0.87, model: fast}")

## Core Concept 2: Model Composition — Chaining Deployments

Ray Serve's killer feature: **composing multiple models** in a pipeline.  
Each model can scale independently, and they communicate via handles (not HTTP).

In [ ]:
COMPOSITION_CODE = '''
# Model composition: Router → [FastModel | AccurateModel]
import ray
from ray import serve
from ray.serve.handle import DeploymentHandle
import numpy as np

@serve.deployment(num_replicas=4)   # scales independently
class FastModel:
    def __init__(self):
        self.model = load_fast_model()

    def predict(self, features: np.ndarray):
        return self.model.predict_proba(features)


@serve.deployment(num_replicas=1, ray_actor_options={"num_gpus": 1})
class AccurateModel:
    def __init__(self):
        self.model = load_accurate_model()

    def predict(self, features: np.ndarray):
        return self.model.predict_proba(features)


@serve.deployment(num_replicas=2)
class Router:
    def __init__(
        self,
        fast: DeploymentHandle,      # handle to FastModel deployment
        accurate: DeploymentHandle,  # handle to AccurateModel deployment
    ):
        self.fast = fast
        self.accurate = accurate

    async def __call__(self, request):
        data = await request.json()
        features = np.array(data["features"]).reshape(1, -1)
        latency_budget = data.get("latency_ms", 100)

        if latency_budget < 10:
            # Tight latency budget → use fast model
            proba = await self.fast.predict.remote(features)
            model = "fast"
        else:
            # More time allowed → use accurate model
            proba = await self.accurate.predict.remote(features)
            model = "accurate"

        pred = int(np.argmax(proba[0]))
        return {"prediction": pred, "confidence": float(max(proba[0])), "model_used": model}


# Wire up composition: Router gets handles to both models
fast_handle     = FastModel.bind()
accurate_handle = AccurateModel.bind()
router          = Router.bind(fast=fast_handle, accurate=accurate_handle)

serve.run(router, route_prefix="/predict")
'''

print("Model composition with Ray Serve:")
print(COMPOSITION_CODE)

print("Key benefits of composition:")
benefits = [
    "Each model scales INDEPENDENTLY (FastModel: 4 replicas, AccurateModel: 1 GPU replica)",
    "In-process communication: no HTTP overhead between models",
    "Async: await handle.method.remote() returns without blocking other requests",
    "Hot update: update one model without restarting others",
    "GPU assignment: GPU model gets exactly the GPU it needs",
]
for b in benefits:
    print(f"  • {b}")

## Core Concept 3: Autoscaling

In [ ]:
autoscaling_config = {
    "min_replicas": 1,          # always have at least 1 replica
    "max_replicas": 10,         # scale up to 10 at peak
    "initial_replicas": 2,
    "target_num_ongoing_requests_per_replica": 5,  # scale up when >5 queued per replica
    "upscale_delay_s": 5,       # wait 5s before scaling up
    "downscale_delay_s": 30,    # wait 30s before scaling down (hysteresis)
}

print("Autoscaling configuration:")
print(json.dumps(autoscaling_config, indent=2))
print()
print("Usage in deployment:")
print("""
  @serve.deployment(
      autoscaling_config={
          'min_replicas': 1,
          'max_replicas': 10,
          'target_num_ongoing_requests_per_replica': 5,
      },
      ray_actor_options={'num_cpus': 1},
  )
  class ScalableModel:
      ...
""")

print("Scaling simulation:")
print("  10 requests/s  → 2 replicas (target: 5 req/replica)")
print("  50 requests/s  → 10 replicas (max reached)")
print("  5 requests/s   → back to 1 replica (after 30s delay)")

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Loading model in `__call__` | Slow every request | Load in `__init__` — runs once per replica |
| Forgetting `await` on handle | Returns a coroutine, not result | Always `await handle.method.remote()` |
| Too many replicas | OOM, slow startup | Match replicas to available CPU/GPU resources |
| `serve.shutdown()` in notebook | Kills all deployments | Use `serve.delete(name)` to remove one deployment |
| Not requesting GPU | GPU model uses CPU | `ray_actor_options={'num_gpus': 1}` |
| Shared mutable state | Race conditions | Ray actors are single-threaded; avoid shared state |

## Interview Questions and Answers

In [ ]:
qa = [
    {"q": "Ray Serve vs BentoML vs FastAPI — when would you choose each?",
     "a": """FastAPI:
- Simple, well-understood, works everywhere
- Great for: simple single-model APIs, teams familiar with FastAPI
- Scaling: horizontal (multiple uvicorn workers or K8s pods)

BentoML:
- ML-specific packaging: model store, containerization, adaptive batching
- Great for: standardized ML serving org-wide, quick containerization
- Scaling: BentoCloud or K8s + BentoML runner

Ray Serve:
- Multi-model composition: chain models in a pipeline, each scales independently
- GPU scheduling: fine-grained GPU allocation per deployment
- Great for: complex serving with routing logic, GPU model farms
- Scaling: Ray cluster (can be 1000s of nodes)

Rule: Single model → FastAPI or BentoML
      Multiple models with complex routing → Ray Serve
      Need model store + containerization → BentoML"""},

    {"q": "How does Ray Serve handle model composition and why is it efficient?",
     "a": """Ray Serve composition uses DeploymentHandles for inter-model communication.

Efficiency reasons:
1. In-process communication: handles use shared memory, no HTTP overhead
   (10-100× faster than HTTP between microservices)

2. Independent scaling: each deployment scales separately
   FastModel: 8 CPU replicas (high throughput)
   AccurateModel: 1 GPU replica (expensive, less traffic)
   Router: 2 replicas (lightweight)

3. Async pipeline: Router awaits models without blocking
   1000 concurrent requests → Router fans out → models run in parallel

4. Fault isolation: if AccurateModel crashes, FastModel still serves

vs REST microservices:
- Separate services: HTTP overhead, service discovery, network latency
- Ray Serve: all in one Ray cluster, shared memory, no network"""},

    {"q": "How does Ray Serve autoscaling work?",
     "a": """Ray Serve autoscaling monitors 'ongoing requests per replica' (queue depth).

Algorithm:
1. Measure: count requests being processed or queued per replica
2. Compare to target_num_ongoing_requests_per_replica
3. If above target → scale up (add replicas), up to max_replicas
4. If below target → scale down (remove replicas), down to min_replicas
5. upscale_delay_s: avoid premature scaling during traffic spikes
6. downscale_delay_s: avoid thrashing (scaling up/down rapidly)

Example:
target = 5 requests/replica
Current: 2 replicas, 30 requests queued → 15/replica >> 5
Action: scale to min(6, max_replicas) replicas

vs K8s HPA:
- K8s HPA scales pods based on CPU/memory
- Ray Serve scales based on request queue depth (more ML-appropriate)
- ML inference: CPU may be idle but model has thousands of queued requests"""},
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 65)
    print()

## Summary

| Concept | Ray Serve API |
|---------|--------------|
| Initialize | `ray.init()` then `serve.start()` |
| Define deployment | `@serve.deployment(num_replicas=N)` on a class |
| GPU allocation | `ray_actor_options={'num_gpus': 1}` |
| Autoscaling | `autoscaling_config={'min': 1, 'max': 10, 'target': 5}` |
| Compose models | `Router.bind(fast=FastModel.bind(), accurate=AccurateModel.bind())` |
| Deploy | `serve.run(deployment, route_prefix='/path')` |
| Call model | `await handle.method.remote(args)` |
| Stop | `serve.shutdown()` |

### Next Steps
1. **Ray Serve docs**: [https://docs.ray.io/en/latest/serve/getting_started.html](https://docs.ray.io/en/latest/serve/getting_started.html)
2. **Ray Serve + FastAPI**: [https://docs.ray.io/en/latest/serve/http-guide.html](https://docs.ray.io/en/latest/serve/http-guide.html)
3. **Next**: Learn Dask and PySpark for processing large datasets efficiently